# WarehouseSort — End-to-End RGB-State Diffusion Policy Solution

This notebook trains and evaluates the end-to-end **RGB-State Diffusion Policy** on all three difficulty levels:
**Easy**, **Medium**, and **Hard**.

Under the updated rules, we include state information (object poses, tag colors, bin positions, and colors) in the observation space, which are automatically flattened into the `obs["state"]` vector. The policy maps this augmented state vector and camera images directly to actions using a Conditional 1D U-Net.

**Requirements:** a CUDA GPU. In Google Colab: *Runtime → Change runtime type → T4 GPU*.

## 1. Installation & Headless Render Setup

In [ ]:
# Install ManiSkill and dependencies (takes ~2 min)
!pip install mani-skill==3.0.1 diffusers==0.38.0 gymnasium torch torchvision hydra-core -q

# Install the warehouse_sort package
!pip install -e . -q

# Colab headless rendering (offscreen Vulkan)
import os
os.environ['DISPLAY'] = ''
os.environ['PYOPENGL_PLATFORM'] = 'egl'

## 2. Level 1: Easy (2 Parcels, Fixed Positions)
In the Easy level, there are 2 parcels with fixed starting poses and fixed bin sides.

In [ ]:
# 1. Generate 200 demonstration episodes (local replay ensures the new observation keys are recorded)
!python il/gen_demos.py --difficulty easy --num-episodes 200 --obs-modes rgb

# 2. Train the RGB-State Diffusion Policy (maps images + augmented state -> actions)
!python il/train.py method=dp_rgb demo_dir=easy \
    flags.total_iters=30000 \
    flags.eval_freq=5000 \
    flags.exp_name=warehouse_rgb_dp_easy

# 3. Evaluate the trained Easy policy
!python eval.py difficulty=easy obs_mode=rgb \
    policy=warehouse_sort.il_policy:load_dp_rgb \
    checkpoint=il/baselines/diffusion_policy/runs/warehouse_rgb_dp_easy/checkpoints/best_eval_sort_accuracy.pt \
    eval_config=conf/eval/default.yaml

## 3. Level 2: Medium (4 Parcels, Jittered Positions)
In the Medium level, there are 4 parcels with small position jitters.

In [ ]:
# 1. Generate 200 demonstration episodes
!python il/gen_demos.py --difficulty medium --num-episodes 200 --obs-modes rgb

# 2. Train the RGB-State Diffusion Policy
!python il/train.py method=dp_rgb demo_dir=medium \
    flags.total_iters=50000 \
    flags.eval_freq=5000 \
    flags.exp_name=warehouse_rgb_dp_medium

# 3. Evaluate the trained Medium policy
!python eval.py difficulty=medium obs_mode=rgb \
    policy=warehouse_sort.il_policy:load_dp_rgb \
    checkpoint=il/baselines/diffusion_policy/runs/warehouse_rgb_dp_medium/checkpoints/best_eval_sort_accuracy.pt \
    eval_config=conf/eval/default.yaml

## 4. Level 3: Hard (6 Parcels, Jittered Positions & Swapping Bins)
In the Hard level, there are 6 parcels and the bin positions swap between episodes.

In [ ]:
# 1. Generate 200 demonstration episodes
!python il/gen_demos.py --difficulty hard --num-episodes 200 --obs-modes rgb

# 2. Train the RGB-State Diffusion Policy
!python il/train.py method=dp_rgb demo_dir=hard \
    flags.total_iters=60000 \
    flags.eval_freq=5000 \
    flags.exp_name=warehouse_rgb_dp_hard

# 3. Evaluate the trained Hard policy
!python eval.py difficulty=hard obs_mode=rgb \
    policy=warehouse_sort.il_policy:load_dp_rgb \
    checkpoint=il/baselines/diffusion_policy/runs/warehouse_rgb_dp_hard/checkpoints/best_eval_sort_accuracy.pt \
    eval_config=conf/eval/default.yaml